In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

# jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 300
seq_len = num_agents * num_time_steps

retnet_embed_dim = 64
retnet_num_heads = 4
num_chunks = 1

2025-02-27 14:30:35.185943: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava-einops/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas *= memory_config.decay_scaling_factor
decay_kappas = jnp.log(decay_kappas)
decay_kappas = decay_kappas[None, :, None, None]

In [3]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=memory_config.decay_scaling_factor,
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros(
    (
        bsz,
        retnet_num_heads,
        retnet_embed_dim // retnet_num_heads,
        retnet_embed_dim // retnet_num_heads,
    )
)
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [5]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
    num_chunks=num_chunks,
)

In [6]:
hstate = copy.deepcopy(init_hstate)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):
    # todo: reset later
    hstate = hstate * jnp.exp(decay_kappas)
    obs_i = obs[:, step * num_agents : (step + 1) * num_agents, ...]
    dones_i = dones[:, step * num_agents : (step + 1) * num_agents]
    step_counts_i = step_counts[:, step * num_agents : (step + 1) * num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, method="recurrent")
    act_output.append(out)

In [7]:
act_output = jnp.concatenate(act_output, axis=1)

In [8]:
act_output.shape

(16, 1200, 64)

In [9]:
hstate = copy.deepcopy(init_hstate)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts, num_chunks=1)

In [10]:
train_out.shape

(16, 1200, 64)

In [11]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(3.8834896e-06, dtype=float32)

In [12]:
jnp.abs(train_out - act_output)

Array([[[4.50760126e-07, 4.53088433e-07, 4.86290082e-06, ...,
         1.90548599e-06, 3.63215804e-08, 3.31364572e-06],
        [7.11902976e-06, 5.43612987e-06, 3.24100256e-07, ...,
         7.55954534e-06, 1.08806416e-05, 3.49618495e-06],
        [7.29854219e-06, 9.48552042e-06, 2.85357237e-06, ...,
         9.96515155e-06, 2.61329114e-06, 7.79796392e-06],
        ...,
        [1.76532194e-06, 7.67968595e-06, 8.07084143e-06, ...,
         2.43959948e-06, 5.45568764e-06, 2.51247548e-06],
        [4.67989594e-06, 7.37351365e-06, 4.76627611e-06, ...,
         1.08219683e-06, 3.60002741e-06, 1.14236027e-05],
        [1.41281635e-06, 2.43368559e-05, 1.93589367e-05, ...,
         1.61398202e-05, 1.02631748e-06, 5.84684312e-06]],

       [[1.79745257e-06, 5.83822839e-07, 5.26430085e-07, ...,
         1.05425715e-06, 7.65873119e-06, 3.94880772e-07],
        [8.48434865e-07, 4.76464629e-06, 1.57672912e-06, ...,
         1.77882612e-07, 1.72853470e-06, 1.83656812e-06],
        [3.16882506e-06, 